# EchoLens R01 — GBT cloud transfer (pilot 1 of 6)

**Status: prepared, not yet executed on cloud data.** One authorized 2019-12-24 NGC6822 HDF5 file (~3.28 GB). Source: [Breakthrough Initiatives Open Data Archive](https://breakthroughinitiatives.org/opendatasearch?file_type=all&order=desc&page=15&perPage=100&project=GBT&search=Search&sort=centerfreq).

This notebook works in **Google Colab**, not on a phone's storage. It inspects the exact archive row, requires a real direct file URL, checks the declared byte length, and stops unless you explicitly set `RUN_DOWNLOAD=True`. It then downloads into **temporary Colab storage**, computes SHA-256 and MD5, uploads to the EchoLens R01 Google Drive folder using the Drive API, and verifies the remote byte count and MD5. Google Drive authentication requires one user approval; credentials never enter GitHub.

This is an *engineering transfer/readback* experiment, **not** a signal detection. Do not treat an ON/OFF label as confirmed until header and cadence verification. Do not run the 19.65 GB six-file collection under this authorization.


In [ ]:
%pip -q install requests beautifulsoup4 google-api-python-client google-auth-httplib2


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urljoin, urlparse
from bs4 import BeautifulSoup
import hashlib, json, os, re, shutil, requests, time

ARCHIVE_PAGE = "https://breakthroughinitiatives.org/opendatasearch?file_type=all&order=desc&page=15&perPage=100&project=GBT&search=Search&sort=centerfreq"
SCAN_START = "2019-12-24 21:09:59"
TARGET = "NGC6822"
EXPECTED_BYTES = 3276424687  # exact bytes shown by public archive, 2019-12-24 21:09:59
DRIVE_FOLDER_ID = "1b6s5yaaNDpjhDtIaobXv1eQ24U6BvXl7"
DIRECT_DOWNLOAD_URL = ""  # if discovery fails, paste URL copied from the archive's download button
RUN_DOWNLOAD = False          # change to True only after reviewing inspection below

def approved_host(url):
    p = urlparse(url)
    host = (p.hostname or '').lower()
    return p.scheme in ('https', 'http') and (
        host.endswith('.berkeley.edu') or host == 'seti.berkeley.edu' or
        host.endswith('.breakthroughinitiatives.org') or host == 'breakthroughinitiatives.org'
    )

def archive_row_candidates(page_html):
    soup = BeautifulSoup(page_html, 'html.parser')
    matches = []
    for tr in soup.find_all('tr'):
        row_text = tr.get_text(' ', strip=True)
        if SCAN_START not in row_text or TARGET not in row_text or str(EXPECTED_BYTES) not in row_text.replace(',', ''):
            continue
        links = []
        for tag in [tr] + list(tr.find_all(True)):
            for k,v in tag.attrs.items():
                if k in ('href', 'data-url', 'data-href', 'download-url', 'onclick'):
                    val = ' '.join(v) if isinstance(v, list) else str(v)
                    parts = re.findall(r'https?://[^\s\"\']+|/[a-zA-Z0-9_./?=&%+-]+', val)
                    if k != 'onclick' and val.startswith(('https://','http://','/')):
                        parts.insert(0,val)
                    for part in parts:
                        url = urljoin(ARCHIVE_PAGE, part.strip('\"\';'))
                        if approved_host(url) and not urlparse(url).path.lower().endswith(('.svg','.png','.jpg','.js','.css')):
                            if url not in links: links.append(url)
        matches.append((row_text, links))
    return matches

resp = requests.get(ARCHIVE_PAGE, timeout=45)
resp.raise_for_status()
matches = archive_row_candidates(resp.text)
assert len(matches) == 1, f'Expected one exact record; found {len(matches)}. STOP: check archive filters.'
print('Verified matching archive row:', matches[0][0])
print('Candidate download URLs:', *(matches[0][1] or ['No direct link extracted: copy the archive Download link into DIRECT_DOWNLOAD_URL']), sep='\n - ')


In [ ]:
# Source selection must be explicit; never substitute a different file.
links = matches[0][1]
source_url = DIRECT_DOWNLOAD_URL.strip() or (links[0] if len(links) == 1 else '')
if not source_url:
    raise RuntimeError('Download URL not uniquely resolved. Copy the exact link from the matched archive row above and set DIRECT_DOWNLOAD_URL, then rerun.')
if not approved_host(source_url):
    raise RuntimeError('Unrecognized download host: refusing transfer. Review archive origin manually.')
if source_url.startswith('http://'):
    print('WARNING: source URL uses HTTP; archive has no published MD5 for this row. Source authenticity cannot be cryptographically verified.')

with requests.Session() as session:
    head = session.head(source_url, allow_redirects=True, timeout=45)
    head_size = int(head.headers.get('Content-Length') or 0) if head.ok else 0
    final_url = head.url if head.ok else source_url
    if not head_size:
        with session.get(source_url, headers={'Range':'bytes=0-0'}, stream=True, timeout=45) as probe:
            probe.raise_for_status()
            final_url = probe.url
            range_header = probe.headers.get('Content-Range','')
            m = re.search(r'/([0-9]+)$', range_header)
            head_size = int(m.group(1)) if m else int(probe.headers.get('Content-Length') or 0)
    if not approved_host(final_url):
        raise RuntimeError('Redirected to an unexpected host: STOP')
    if head_size != EXPECTED_BYTES:
        raise RuntimeError(f'Wrong file length: archive={EXPECTED_BYTES}, server={head_size}. STOP')
print('Preflight passed: known target, approved host, exact byte length:', head_size)
print('Host:', urlparse(final_url).hostname)
print('Download execution enabled:', RUN_DOWNLOAD)


In [ ]:
if not RUN_DOWNLOAD:
    print('SAFE STOP: no data transferred. Set RUN_DOWNLOAD=True in configuration and rerun this cell and the next cell only when ready.')
else:
    from google.colab import auth
    import google.auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaFileUpload, MediaInMemoryUpload

    auth.authenticate_user()  # one-time Google authorization within your own Colab session
    creds, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive'])
    drive = build('drive', 'v3', credentials=creds)
    folder = drive.files().get(fileId=DRIVE_FOLDER_ID, fields='id,name,mimeType', supportsAllDrives=True).execute()
    assert folder['mimeType'] == 'application/vnd.google-apps.folder', 'Drive destination is not a folder'
    print('Destination verified:', folder['name'])

    local = Path('/content/echolens_r01/GBT_NGC6822_20191224_210959_UTC.h5')
    local.parent.mkdir(parents=True, exist_ok=True)
    assert shutil.disk_usage(local.parent).free > EXPECTED_BYTES + 1024**3, 'Colab disk insufficient; STOP'
    temp = local.with_suffix('.h5.part')
    sha, md5, count = hashlib.sha256(), hashlib.md5(), 0
    with requests.get(source_url, stream=True, timeout=(30,120), allow_redirects=True) as stream:
        stream.raise_for_status()
        assert approved_host(stream.url), 'Download redirected off the approved archive host list'
        with temp.open('wb') as fp:
            for chunk in stream.iter_content(chunk_size=8 * 1024 * 1024):
                if not chunk: continue
                count += len(chunk)
                if count > EXPECTED_BYTES: raise ValueError('Exceeded expected bytes; STOP')
                fp.write(chunk); sha.update(chunk); md5.update(chunk)
    if count != EXPECTED_BYTES: raise ValueError(f'Truncated: {count} != {EXPECTED_BYTES}; STOP')
    os.replace(temp, local)
    sha_hex, md5_hex = sha.hexdigest(), md5.hexdigest()
    print('Local validated bytes and hashes:', count, sha_hex, md5_hex)

    q = f"name = '{local.name}' and '{DRIVE_FOLDER_ID}' in parents and trashed = false"
    existing = drive.files().list(q=q, fields='files(id,name,size,md5Checksum),nextPageToken', supportsAllDrives=True).execute().get('files',[])
    if existing:
        matches_existing = [f for f in existing if int(f.get('size',-1)) == EXPECTED_BYTES and f.get('md5Checksum') == md5_hex]
        if len(existing) != 1 or len(matches_existing) != 1: raise RuntimeError('Conflicting Drive filename; refusing overwrite/duplicate')
        stored = matches_existing[0]
    else:
        media = MediaFileUpload(str(local), mimetype='application/x-hdf5', chunksize=16*1024*1024, resumable=True)
        request = drive.files().create(body={'name':local.name,'parents':[DRIVE_FOLDER_ID]}, media_body=media, fields='id,name,size,md5Checksum,webViewLink', supportsAllDrives=True)
        stored = None
        while stored is None:
            _progress, stored = request.next_chunk(num_retries=4)
    result = drive.files().get(fileId=stored['id'],fields='id,name,size,md5Checksum,webViewLink',supportsAllDrives=True).execute()
    assert int(result['size']) == count and result.get('md5Checksum') == md5_hex, 'Drive upload checksum/length mismatch'
    manifest = {
        'agent':'R01', 'stage':'ONE_FILE_ACQUIRED_UNINSPECTED', 'is_detection':False,
        'original_archive_page':ARCHIVE_PAGE, 'target_label':TARGET, 'scan_start_utc':SCAN_START.replace(' ','T')+'Z',
        'expected_and_verified_bytes':count, 'sha256':sha_hex, 'md5':md5_hex,
        'source_host':urlparse(final_url).hostname, 'archive_checksum':None,
        'drive_file_id':result['id'], 'drive_file_name':result['name'],
        'retrieved_at_utc':datetime.now(timezone.utc).isoformat(),
        'known_limitations':['No independent archive checksum for this row', 'ON/OFF relation still provisional', 'HDF5 header not yet inspected']
    }
    manifest_bytes = json.dumps(manifest,indent=2).encode('utf-8')
    manifest_media = MediaInMemoryUpload(manifest_bytes,mimetype='application/json',resumable=False)
    drive.files().create(body={'name':'R01A_NGC6822_first_scan_manifest.json','parents':[DRIVE_FOLDER_ID]},media_body=manifest_media,fields='id,name',supportsAllDrives=True).execute()
    print('Cloud transfer complete; verified Drive object:',result)
    print('Manifest:',json.dumps(manifest,indent=2))


## Next scientific gate

Only after the transfer succeeds: record the file IDs, SHA-256 and Drive integrity result in [Issue #16](https://github.com/gittenssexton/echolens-api/issues/16). Then run a **separate**, bounded HDF5 header/readback QA (sample and full band metadata; polarization, integration time, channel width and frequency reference) before any candidate search or synthetic injections. No discoveries are implied by successful transfer.
